# Day 5 — Retrieval evaluation + alpha sweep

**Goal:** evaluate retrieval quality (Recall@K) on positive-labeled cases and tune alpha for best Recall@5.

In [ ]:
!pip -q install pandas tqdm faiss-cpu

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Load dataset + CheXpert labels and build positive subset

In [ ]:

import faiss

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
chex = pd.read_csv(f"{RAW}/mimic-cxr-2.0.0-chexpert.csv.gz")

# Merge labels
label_cols = ["Cardiomegaly","Pleural Effusion","Pneumothorax","Pneumonia","Consolidation","No Finding","Atelectasis"]
keep = ["subject_id","study_id"] + [c for c in label_cols if c in chex.columns]
chex = chex[keep].copy()

df_lab = df.merge(chex, on=["subject_id","study_id"], how="left")
print("After merge shape:", df_lab.shape)
missing = df_lab[label_cols].isna().all(axis=1).sum() if all(c in df_lab.columns for c in label_cols) else "n/a"
print("Missing label rows:", missing)

# Define positives as any of selected findings == 1
findings = [c for c in ["Cardiomegaly","Pleural Effusion","Pneumothorax","Pneumonia","Consolidation"] if c in df_lab.columns]
df_pos = df_lab[df_lab[findings].fillna(0).sum(axis=1) > 0].copy()
print("Rows with at least one positive finding:", len(df_pos))


## 2) Utility: build index and compute Recall@K

In [ ]:

def build_index(emb):
    emb = emb.astype("float32")
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    return idx, emb

def recall_at_k(index, emb_all, df_all, df_pos, k=5, n_eval=300):
    # Evaluate first n_eval positive rows
    n_eval = min(n_eval, len(df_pos))
    hits = 0
    for j in tqdm(range(n_eval)):
        # row_idx in df_all space
        pos_row = df_pos.iloc[j]
        # locate its index in df_all
        # safest: match by study_id AND image_path
        mask = (df_all["study_id"] == pos_row["study_id"]) & (df_all["image_path"] == pos_row["image_path"])
        q_idx = int(np.where(mask.values)[0][0])

        q = emb_all[q_idx:q_idx+1]
        scores, inds = index.search(q, k+1)  # includes itself
        inds = [i for i in inds[0].tolist() if i != q_idx][:k]

        # hit if any retrieved shares at least one positive finding with query
        q_lab = pos_row[findings].fillna(0).values.astype(int)
        ok = False
        for i in inds:
            cand = df_all.iloc[i]
            cand_lab = cand[findings].fillna(0).values.astype(int)
            if (q_lab * cand_lab).sum() > 0:
                ok = True
                break
        hits += int(ok)
    return hits / n_eval



## 3) Evaluate image-only vs fused (alpha=0.5)

In [ ]:

image_emb = np.load(f"{BASE}/image_embeddings_2696.npy").astype("float32")
fused = np.load(f"{BASE}/fused_embeddings_alpha_0.5.npy").astype("float32")

# Align df_all with labels
df_all = df_lab.copy()

idx_img, emb_img = build_index(image_emb)
idx_fused, emb_fused = build_index(fused)

r5_img = recall_at_k(idx_img, emb_img, df_all, df_pos, k=5, n_eval=300)
r5_fused = recall_at_k(idx_fused, emb_fused, df_all, df_pos, k=5, n_eval=300)

print("Recall@5 image-only:", r5_img)
print("Recall@5 fusion(alpha=0.5):", r5_fused)


## 4) Alpha sweep for best Recall@5

In [ ]:

# If you saved text embeddings separately, load them; otherwise recompute day4 and save as text_embeddings_2696.npy
text_path = f"{BASE}/text_embeddings_2696.npy"
if not os.path.exists(text_path):
    raise FileNotFoundError("Missing text embeddings. Run Day 4 and save text_embeddings_2696.npy first.")
text_emb = np.load(text_path).astype("float32")

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
results = []
for a in alphas:
    fused_a = a*image_emb + (1-a)*text_emb
    fused_a = fused_a / np.linalg.norm(fused_a, axis=1, keepdims=True)
    idx_a, emb_a = build_index(fused_a.astype("float32"))
    r5 = recall_at_k(idx_a, emb_a, df_all, df_pos, k=5, n_eval=300)
    results.append({"alpha": a, "recall@5": r5})
    print("alpha:", a, "recall@5:", r5)

res_df = pd.DataFrame(results).sort_values("recall@5", ascending=False)
best = res_df.iloc[0].to_dict()
print("Best alpha:", best["alpha"], "Best Recall@5:", best["recall@5"])

res_df.to_csv(f"{BASE}/alpha_sweep_recall5.csv", index=False)
print("Saved:", f"{BASE}/alpha_sweep_recall5.csv")
res_df


✅ **Day 5 deliverables:** Recall@K numbers + `alpha_sweep_recall5.csv`